# Raw Code vs Transformer Embedding Downstream Comparison (v2)

**Goal:** Compare downstream IP prediction performance across dimension-reduction
approaches applied to raw medical codes (~39k active features):

| # | Method | Type |
|---|--------|------|
| 1 | **Correlation Analysis** | Per-tier association of raw codes with IP outcomes |
| 2 | **PCA(256)** | Linear reduction via TruncatedSVD |
| 3 | **AutoEncoder(256)** | Nonlinear compression via 4-GPU neural network |
| 4 | **SelectKBest(256, chi2)** | Univariate filter selection (sparse-native) |
| 5 | **TE Embedding(256)** | Pretrained transformer encoder (R6 best model) |

**Reference:** `moe_flashattn_3_lob3_downstream_running.py` — same CatBoost config, splits, metrics.

**Structure:**
- **Part A** (run once): Generate and cache base artifacts from BigQuery
- **Part B** (run from cache): Correlation analysis, dimension reduction, downstream evaluation, visualization

In [ ]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from scipy.sparse import lil_matrix
from scipy.stats import t as t_dist
from collections import Counter
from typing import Dict, List, Tuple, Optional

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

from catboost import CatBoostClassifier, Pool
import shap

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import google.auth
from google.cloud import bigquery

In [ ]:
# ============================================================================
# CONFIGURATION — all user-tunable parameters in one place
# ============================================================================

PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"

RAW_TE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a834793_Combined_All_LOB_o3_train_ending"
FEATURES_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "a964286_commercial_ip_heldout_transformer_matched_final_dataset_4_te_experiment_round5_downstream"
)
TE_EMBEDDING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "a964286_te4exp_3lob_exp_round5_v2_exp2b_flash_learned_pool_asym_focalloss_densesampler_commercial_all_sample_embedding"
)

TARGET_COLUMN = "ip6"
OOT_CUTOFF_DATE = "2023-10-16"
TARGET_DIM = 256
RANDOM_STATE = 42
NEGATIVE_DOWNSAMPLE_RATIO = 10

# CatBoost — matches moe_flashattn_3_lob3_downstream_running.py line 2150
CATBOOST_PARAMS = {
    'iterations': 2500,
    'depth': 7,
    'learning_rate': 0.025,
    'grow_policy': 'SymmetricTree',
    'auto_class_weights': 'Balanced',
    'od_wait': 80,
    'use_best_model': True,
    'random_seed': RANDOM_STATE,
    'verbose': 0,
}

# Autoencoder config
AE_HIDDEN_DIMS = [2048, 512]
AE_EPOCHS = 15
AE_BATCH_SIZE = 2048
AE_LR = 1e-3
AE_WEIGHT_DECAY = 1e-5

TIER_NAMES = ['common', 'medium', 'rare', 'tail']

OUTPUT_DIR = "downstream_eval/raw_code_vs_te_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Utility Functions

Metrics, splitting, downsampling — consistent with `moe_flashattn_3_lob3_downstream_running.py`.

In [ ]:
def lift_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    k = max(1, int(len(y_true) * pct))
    top_k = np.argsort(y_prob)[::-1][:k]
    baseline = y_true.mean()
    return y_true[top_k].mean() / baseline if baseline > 0 else 0.0


def true_positives_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> int:
    k = max(1, int(len(y_true) * pct))
    return int(y_true[np.argsort(y_prob)[::-1][:k]].sum())


def precision_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    k = max(1, int(len(y_true) * pct))
    return float(y_true[np.argsort(y_prob)[::-1][:k]].mean())


def compute_split_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    return {
        'auc_roc': roc_auc_score(y_true, y_prob),
        'auc_pr': average_precision_score(y_true, y_prob),
        'brier': brier_score_loss(y_true, y_prob),
        'lift_1pct': lift_at_percentage(y_true, y_prob, 0.01),
        'lift_5pct': lift_at_percentage(y_true, y_prob, 0.05),
        'lift_10pct': lift_at_percentage(y_true, y_prob, 0.10),
        'tp_1pct': true_positives_at_percentage(y_true, y_prob, 0.01),
        'precision_1pct': precision_at_percentage(y_true, y_prob, 0.01),
        'n_samples': len(y_true),
        'n_positives': int(y_true.sum()),
        'prevalence': float(y_true.mean()),
    }

In [ ]:
EXCLUDE_COLUMNS = frozenset([
    'individual_id', 'member_id', 'index_dt', 'birth_dt', 'feature_end_dt',
    'ip6', 'sum_ip6_admits', 'sum_ip6_los', 'sum_ip6_acu_days',
    'mon_3_include', 'mon_6_include', 'mon_12_include',
    'exclude_ip', 'include_post_6_status', 'ind_id_last_digit',
    'clm_allowed_amt_1yr', 'clm_allowed_amt_2yr', 'clm_allowed_amt_3mo', 'clm_allowed_amt_6mo',
    'clm_paid_amt_1yr', 'clm_paid_amt_2yr', 'clm_paid_amt_3mo', 'clm_paid_amt_6mo',
    'clm_par_allowed_amt_1yr', 'clm_par_allowed_amt_2yr', 'clm_par_allowed_amt_3mo', 'clm_par_allowed_amt_6mo',
    'clm_par_paid_amt_1yr', 'clm_par_paid_amt_2yr', 'clm_par_paid_amt_3mo', 'clm_par_paid_amt_6mo',
    'clm_srv_copay_amt_1yr', 'clm_srv_copay_amt_3mo', 'clm_srv_copay_amt_6mo',
    'covid_19', 'hpd_major_flag', 'chronic',
    'txt_member', 'txt_referral', 'txt_1yr_outreach', 'talked'
])


def create_data_splits(df: pd.DataFrame, oot_cutoff_date: str = OOT_CUTOFF_DATE) -> Dict[str, pd.DataFrame]:
    df = df.copy()
    df['_dt'] = pd.to_datetime(df['index_dt'])
    cutoff = pd.to_datetime(oot_cutoff_date)
    splits = {
        'train': df[(df['ind_id_last_digit'].isin(range(8))) & (df['_dt'] <= cutoff)],
        'val':   df[(df['ind_id_last_digit'] == 8) & (df['_dt'] <= cutoff)],
        'test':  df[(df['ind_id_last_digit'] == 9) & (df['_dt'] <= cutoff)],
        'oot':   df[df['_dt'] > cutoff],
        'oot_strict': df[(df['_dt'] > cutoff) & (df['ind_id_last_digit'] == 9)],
    }
    print("Data splits:")
    for name, sdf in splits.items():
        if len(sdf) > 0:
            prev = sdf[TARGET_COLUMN].mean() * 100
            print(f"  {name}: {len(sdf):,} rows, {int(sdf[TARGET_COLUMN].sum()):,} pos ({prev:.2f}%)")
        else:
            print(f"  {name}: EMPTY")
    return {k: v.drop(columns=['_dt']) for k, v in splits.items()}


def downsample_negatives(
    X: pd.DataFrame, y: pd.Series,
    ratio: int = NEGATIVE_DOWNSAMPLE_RATIO, random_state: int = RANDOM_STATE,
) -> Tuple[pd.DataFrame, pd.Series]:
    rng = np.random.RandomState(random_state)
    pos_idx = X.index[y == 1].tolist()
    neg_idx = X.index[y == 0].tolist()
    target_neg = int(len(pos_idx) * ratio)
    if len(neg_idx) <= target_neg:
        return X, y
    sampled = rng.choice(neg_idx, size=target_neg, replace=False).tolist()
    keep = pos_idx + sampled
    X_r, y_r = X.loc[keep].copy(), y.loc[keep].copy()
    order = rng.permutation(len(X_r))
    X_r = X_r.iloc[order].reset_index(drop=True)
    y_r = y_r.iloc[order].reset_index(drop=True)
    print(f"  Downsample: {len(neg_idx):,}->{target_neg:,} neg, {len(pos_idx):,} pos ({ratio}:1)")
    return X_r, y_r

In [ ]:
class ArtifactCache:
    """Registry-based artifact cache with typed load/save and status display."""

    _REGISTRY = {
        'code_matrix_full':    ('te_row_code_frequency_matrix.npz',            'sparse'),
        'member_ids':          ('te_row_member_ids_ordered.npy',               'numpy'),
        'tier_data':           ('code_frequency_tiers.json',                   'json'),
        'df_merged_base':      ('df_merged_base.parquet',                      'parquet'),
        'code_matrix_active':  ('code_matrix_active.npz',                     'sparse'),
        'active_code_indices': ('active_code_indices.npy',                    'numpy'),
        'df_features':         ('df_features.parquet',                         'parquet'),
        'pca_features':        ('te_vs_raw_code_pca_256_features.npy',        'numpy'),
        'ae_features':         (f'ae_{TARGET_DIM}_features.npy',               'numpy'),
        'ae_model':            (f'ae_{TARGET_DIM}_model.pt',                   'torch'),
        'selectk_features':    (f'selectk_chi2_{TARGET_DIM}_features.npy',     'numpy'),
        'selectk_indices':     (f'selectk_chi2_{TARGET_DIM}_code_indices.npy', 'numpy'),
        'selectk_scores':      (f'selectk_chi2_{TARGET_DIM}_scores.npy',       'numpy'),
        'te_embeddings':       ('te_embeddings.parquet',                       'parquet'),
        'correlation_results': ('tier_correlation_analysis.parquet',           'parquet'),
        'comparison_results':  ('te_vs_raw_code_comparison_results.csv',       'csv'),
    }

    def __init__(self, base_dir: str):
        self.base_dir = base_dir

    def path(self, name: str) -> str:
        return os.path.join(self.base_dir, self._REGISTRY[name][0])

    def exists(self, name: str) -> bool:
        return os.path.exists(self.path(name))

    def _size_mb(self, name: str) -> float:
        p = self.path(name)
        return os.path.getsize(p) / 1e6 if os.path.exists(p) else 0.0

    def save(self, name: str, data):
        fmt = self._REGISTRY[name][1]
        p = self.path(name)
        if   fmt == 'sparse':  sparse.save_npz(p, data)
        elif fmt == 'numpy':   np.save(p, data)
        elif fmt == 'json':
            with open(p, 'w') as f: json.dump(data, f, indent=2)
        elif fmt == 'parquet': data.to_parquet(p, index=False)
        elif fmt == 'csv':     data.to_csv(p, index=False)
        elif fmt == 'torch':   torch.save(data, p)
        print(f"  [SAVED] {os.path.basename(p)} ({self._size_mb(name):.1f} MB)")

    def load(self, name: str):
        fmt = self._REGISTRY[name][1]
        p = self.path(name)
        if   fmt == 'sparse':  return sparse.load_npz(p)
        elif fmt == 'numpy':   return np.load(p, allow_pickle=True)
        elif fmt == 'json':
            with open(p) as f: return json.load(f)
        elif fmt == 'parquet': return pd.read_parquet(p)
        elif fmt == 'csv':     return pd.read_csv(p)
        elif fmt == 'torch':   return torch.load(p, map_location='cpu', weights_only=False)

    def status(self, names: Optional[List[str]] = None):
        names = names or list(self._REGISTRY.keys())
        print(f"{'Artifact':<24} {'File':<50} {'Status'}")
        print("-" * 95)
        for name in names:
            fname = self._REGISTRY[name][0]
            if self.exists(name):
                print(f"  {name:<22} {fname:<48} cached  {self._size_mb(name):>7.1f} MB")
            else:
                print(f"  {name:<22} {fname:<48} missing")


cache = ArtifactCache(OUTPUT_DIR)

In [ ]:
class SparseRowDataset(Dataset):
    """Converts sparse matrix rows to dense tensors on-the-fly for GPU training."""

    def __init__(self, sparse_matrix):
        self.data = sparse_matrix

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx].toarray().ravel().astype(np.float32))


class CodeAutoEncoder(nn.Module):
    """Symmetric autoencoder with configurable bottleneck and hidden dimensions."""

    def __init__(self, input_dim: int, bottleneck_dim: int = 256,
                 hidden_dims: Optional[List[int]] = None):
        super().__init__()
        hidden_dims = hidden_dims or [2048, 512]

        enc, prev = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(True), nn.Dropout(0.2)]
            prev = h
        enc.append(nn.Linear(prev, bottleneck_dim))
        self.encoder = nn.Sequential(*enc)

        dec, prev = [], bottleneck_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(True)]
            prev = h
        dec += [nn.Linear(prev, input_dim), nn.ReLU()]
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def encode(self, x):
        return self.encoder(x)


def train_autoencoder(
    sparse_matrix,
    bottleneck_dim: int = TARGET_DIM,
    hidden_dims: Optional[List[int]] = None,
    epochs: int = AE_EPOCHS,
    batch_size: int = AE_BATCH_SIZE,
    lr: float = AE_LR,
):
    """Train autoencoder on sparse code-frequency matrix using up to 4 GPUs."""
    hidden_dims = hidden_dims or AE_HIDDEN_DIMS
    input_dim = sparse_matrix.shape[1]

    dataset = SparseRowDataset(sparse_matrix)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=True,
    )

    model = CodeAutoEncoder(input_dim, bottleneck_dim, hidden_dims)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_gpus = min(4, torch.cuda.device_count()) if torch.cuda.is_available() else 0

    if n_gpus > 1:
        print(f"DataParallel: {n_gpus} GPUs")
        model = nn.DataParallel(model, device_ids=list(range(n_gpus)))
    elif n_gpus == 1:
        print("Single GPU")
    else:
        print("CPU only")

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=AE_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    arch = f"{input_dim} -> {' -> '.join(map(str, hidden_dims))} -> {bottleneck_dim}"
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Architecture: {arch} (mirror decoder)")
    print(f"Parameters: {total_params:,} | Samples: {sparse_matrix.shape[0]:,} | Batches/epoch: {len(loader)}")

    for epoch in range(epochs):
        model.train()
        total_loss, n_batch = 0.0, 0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for batch in pbar:
            batch = batch.to(device, non_blocking=True)
            x_hat, _ = model(batch)
            loss = nn.functional.mse_loss(x_hat, batch)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batch += 1
            pbar.set_postfix(loss=f"{loss.item():.6f}")
        scheduler.step()
        print(f"  Epoch {epoch+1}/{epochs}: loss={total_loss/n_batch:.6f}, lr={scheduler.get_last_lr()[0]:.2e}")

    return model


@torch.no_grad()
def encode_all(model, sparse_matrix, batch_size: int = 4096) -> np.ndarray:
    """Encode entire sparse matrix through the trained autoencoder's encoder."""
    model.eval()
    device = next(model.parameters()).device
    loader = DataLoader(
        SparseRowDataset(sparse_matrix), batch_size=batch_size,
        shuffle=False, num_workers=4, pin_memory=True,
    )
    parts = []
    for batch in tqdm(loader, desc="Encoding"):
        batch = batch.to(device, non_blocking=True)
        raw = model.module if hasattr(model, 'module') else model
        parts.append(raw.encode(batch).cpu().numpy())
    return np.concatenate(parts, axis=0)

In [ ]:
def compute_tier_correlations(
    code_matrix, y, code_tiers: dict, active_code_indices: np.ndarray,
) -> pd.DataFrame:
    """
    Vectorized point-biserial correlation of each code with the binary target,
    grouped by frequency tier. Fully vectorized — no per-feature loops for computation.
    """
    y = np.asarray(y, dtype=np.float64)
    n = len(y)
    n1, n0 = y.sum(), n - y.sum()
    pos_mask = y == 1

    X = code_matrix.tocsc()
    mean_1 = np.array(X[pos_mask].mean(axis=0)).ravel()
    mean_0 = np.array(X[~pos_mask].mean(axis=0)).ravel()
    mean_all = np.array(X.mean(axis=0)).ravel()

    var_all = np.array(X.multiply(X).mean(axis=0)).ravel() - mean_all ** 2
    std_all = np.sqrt(np.maximum(var_all, 1e-15))

    r = (mean_1 - mean_0) / std_all * np.sqrt(n0 * n1) / n
    r_clip = np.clip(r, -0.9999, 0.9999)
    t_stat = r_clip * np.sqrt((n - 2) / (1 - r_clip ** 2))
    p_vals = 2 * t_dist.sf(np.abs(t_stat), df=n - 2)

    rows = []
    for i, cidx in enumerate(active_code_indices):
        rows.append({
            'code_idx': int(cidx),
            'tier': code_tiers.get(int(cidx), 'unknown'),
            'correlation': r[i],
            'abs_correlation': abs(r[i]),
            'p_value': p_vals[i],
            'significant_001': p_vals[i] < 0.001,
            'significant_01':  p_vals[i] < 0.01,
            'significant_05':  p_vals[i] < 0.05,
            'mean_freq_pos': mean_1[i],
            'mean_freq_neg': mean_0[i],
            'freq_ratio': mean_1[i] / max(mean_0[i], 1e-10),
        })
    return pd.DataFrame(rows)


def evaluate_feature_set(
    feature_matrix: np.ndarray,
    feature_names: list,
    df_base: pd.DataFrame,
    feature_set_name: str,
) -> dict:
    """
    Full downstream evaluation: split -> downsample -> CatBoost -> metrics.
    Same pipeline as moe_flashattn_3_lob3_downstream_running.py.
    """
    print(f"\n{'='*70}")
    print(f"Evaluating: {feature_set_name} ({feature_matrix.shape[1]} features)")
    print(f"{'='*70}")

    df_eval = df_base[[
        'individual_id', 'index_dt', TARGET_COLUMN, 'ind_id_last_digit'
    ]].copy().reset_index(drop=True)
    assert len(feature_matrix) == len(df_eval), \
        f"Row mismatch: features={len(feature_matrix)}, base={len(df_eval)}"

    feat_df = pd.DataFrame(feature_matrix, columns=feature_names, dtype=np.float32)
    df_eval = pd.concat([df_eval, feat_df], axis=1)
    splits = create_data_splits(df_eval)

    X_sp, y_sp = {}, {}
    for s, sdf in splits.items():
        if len(sdf) > 0:
            X_sp[s] = sdf[feature_names].copy()
            y_sp[s] = sdf[TARGET_COLUMN].astype(int)

    if 'train' in X_sp:
        X_sp['train'], y_sp['train'] = downsample_negatives(X_sp['train'], y_sp['train'])

    model = CatBoostClassifier(**CATBOOST_PARAMS)
    model.fit(
        Pool(X_sp['train'], y_sp['train']),
        eval_set=Pool(X_sp['val'], y_sp['val']),
        verbose=0,
    )

    results = {'feature_set': feature_set_name, 'n_features': feature_matrix.shape[1]}
    for split in ['val', 'test', 'oot', 'oot_strict']:
        if split not in X_sp:
            continue
        probs = model.predict_proba(X_sp[split])[:, 1]
        m = compute_split_metrics(np.array(y_sp[split]), probs)
        for k, v in m.items():
            results[f'{split}_{k}'] = v
        print(f"  {split}: AUC={m['auc_roc']:.4f}, "
              f"Lift@1%={m['lift_1pct']:.2f}, Lift@5%={m['lift_5pct']:.2f}")

    results['model'] = model
    results['X_test'] = X_sp.get('test')
    results['y_test'] = y_sp.get('test')
    return results

---
## Part A: Data Generation (Run Once, Then Skip to Part B)

These cells load raw data from BigQuery, build the sparse code-frequency matrix,
and save cached artifacts. **Skip this entire section if artifacts already exist locally.**

In [ ]:
# ========================  A.1: Load from BigQuery  ========================
client = bigquery.Client()

# Downstream features table
df_features = client.query(f"SELECT * FROM `{FEATURES_TABLE}`").to_dataframe()
df_features['individual_id'] = df_features['individual_id'].astype(str)
print(f"Features: {len(df_features):,} rows, {len(df_features.columns)} cols, "
      f"prevalence={df_features[TARGET_COLUMN].mean()*100:.2f}%")
cache.save('df_features', df_features)

# Raw code sequences for Commercial members
df_raw = client.query(f"""
    SELECT individual_id, cd, dt_cnt
    FROM `{RAW_TE_TABLE}` WHERE lob = 'Commercial'
""").to_dataframe()
df_raw['individual_id'] = df_raw['individual_id'].astype(str)
downstream_ids = set(df_features['individual_id'])
df_raw_matched = df_raw[df_raw['individual_id'].isin(downstream_ids)].copy()
del df_raw
print(f"Matched raw sequences: {len(df_raw_matched):,} members")

# TE embeddings
df_te = client.query(f"SELECT * FROM `{TE_EMBEDDING_TABLE}`").to_dataframe()
df_te['individual_id'] = df_te['individual_id'].astype(str)
cache.save('te_embeddings', df_te)
print(f"TE embeddings: {len(df_te):,} rows")

In [ ]:
# ========================  A.2: Parse Codes -> Sparse Matrix  ========================

def parse_cd_to_freq(cd_string: str) -> Counter:
    """Parse 'code1,code2*code3,code4*...' into Counter(code -> total_occurrences)."""
    if not cd_string or pd.isna(cd_string):
        return Counter()
    freq = Counter()
    for day in cd_string.split('*'):
        for tok in day.split(','):
            try:
                code = int(tok)
                if code > 0:
                    freq[code] += 1
            except ValueError:
                continue
    return freq


member_freqs, member_ids_ordered = [], []
for _, row in tqdm(df_raw_matched.iterrows(), total=len(df_raw_matched), desc="Parsing codes"):
    member_freqs.append(parse_cd_to_freq(row['cd']))
    member_ids_ordered.append(row['individual_id'])

all_codes = set().union(*member_freqs)
vocab_size = max(all_codes) + 1 if all_codes else 0
print(f"Active codes: {len(all_codes):,}, vocab size: {vocab_size:,}")

code_matrix = lil_matrix((len(member_freqs), vocab_size), dtype=np.float32)
for i, freq in tqdm(enumerate(member_freqs), total=len(member_freqs), desc="Building sparse"):
    for code, count in freq.items():
        code_matrix[i, code] = count
code_matrix_csr = code_matrix.tocsr()
del code_matrix

cache.save('code_matrix_full', code_matrix_csr)
cache.save('member_ids', np.array(member_ids_ordered))
print(f"Sparse: {code_matrix_csr.shape}, nnz={code_matrix_csr.nnz:,}")

In [ ]:
# ========================  A.3: Frequency Tier Analysis  ========================

code_occurrence_counts = np.array(code_matrix_csr.sum(axis=0)).ravel()
active_codes = np.where(code_occurrence_counts > 0)[0]
active_freqs = code_occurrence_counts[active_codes]
freq_pcts = np.percentile(active_freqs, [20, 50, 80])
print(f"Tier thresholds: p20={freq_pcts[0]:.0f}, p50={freq_pcts[1]:.0f}, p80={freq_pcts[2]:.0f}")

code_tiers = {}
for cidx in active_codes:
    f = code_occurrence_counts[cidx]
    if   f > freq_pcts[2]: code_tiers[int(cidx)] = 'common'
    elif f > freq_pcts[1]: code_tiers[int(cidx)] = 'medium'
    elif f > freq_pcts[0]: code_tiers[int(cidx)] = 'rare'
    else:                  code_tiers[int(cidx)] = 'tail'

tier_counts = Counter(code_tiers.values())
for t in TIER_NAMES:
    print(f"  {t}: {tier_counts.get(t, 0):,} codes")

tier_data = {
    'percentile_thresholds': {
        'p20': float(freq_pcts[0]), 'p50': float(freq_pcts[1]), 'p80': float(freq_pcts[2]),
    },
    'tier_counts': dict(tier_counts),
    'code_tiers': {str(k): v for k, v in code_tiers.items()},
    'n_active_codes': len(active_codes),
}
cache.save('tier_data', tier_data)

In [ ]:
# ========================  A.4: Merge + Active Matrix  ========================

df_code_ids = pd.DataFrame({
    'individual_id': member_ids_ordered,
    '_sparse_row_idx': range(len(member_ids_ordered)),
})
df_code_ids['individual_id'] = df_code_ids['individual_id'].astype(str)

df_base = df_features[[
    'individual_id', 'index_dt', TARGET_COLUMN, 'ind_id_last_digit'
]].copy()
df_base['individual_id'] = df_base['individual_id'].astype(str)

df_merged_base = df_base.merge(df_code_ids, on='individual_id', how='inner')
df_merged_base = df_merged_base.drop_duplicates(
    subset=['individual_id', 'index_dt'], keep='last'
)
print(f"Merged base: {len(df_merged_base):,} rows")

code_matrix_matched = code_matrix_csr[df_merged_base['_sparse_row_idx'].values]
active_mask = np.array(code_matrix_matched.sum(axis=0)).ravel() > 0
code_matrix_active = code_matrix_matched[:, active_mask]
active_code_indices = np.where(active_mask)[0]
print(f"Active features: {code_matrix_active.shape[1]:,} "
      f"(removed {(~active_mask).sum():,} zero columns)")

cache.save('df_merged_base', df_merged_base)
cache.save('code_matrix_active', code_matrix_active)
cache.save('active_code_indices', active_code_indices)

---
## Part B: Load Cached Artifacts & Run Analysis

All cells below assume Part A artifacts exist locally. Start here on subsequent runs.

In [ ]:
# ========================  B.0: Load All Cached Artifacts  ========================
t0 = time.time()

code_matrix_active = cache.load('code_matrix_active')
active_code_indices = cache.load('active_code_indices')
df_merged_base = cache.load('df_merged_base')
df_merged_base['individual_id'] = df_merged_base['individual_id'].astype(str)

tier_data = cache.load('tier_data')
code_tiers = {int(k): v for k, v in tier_data['code_tiers'].items()}
freq_percentiles = np.array([
    tier_data['percentile_thresholds']['p20'],
    tier_data['percentile_thresholds']['p50'],
    tier_data['percentile_thresholds']['p80'],
])

active_code_names = [f'code_{i}' for i in active_code_indices]
n_members = code_matrix_active.shape[0]
y_all = df_merged_base[TARGET_COLUMN].values.astype(int)

print(f"Loaded in {time.time()-t0:.1f}s:")
print(f"  code_matrix_active: {code_matrix_active.shape}, nnz={code_matrix_active.nnz:,}")
print(f"  df_merged_base: {len(df_merged_base):,} rows")
print(f"  Tiers: {dict(Counter(code_tiers.values()))}")
print(f"  Target prevalence: {y_all.mean()*100:.2f}%")
print()
cache.status()

### B.1 Correlation Analysis: Code Frequency Tiers vs IP Outcomes

Vectorized point-biserial correlation for all active codes with the binary IP target.
Goal: understand what proportion of codes at each frequency tier is significantly
associated with IP outcomes.

In [ ]:
if cache.exists('correlation_results'):
    print("Loading cached correlation results...")
    df_corr = cache.load('correlation_results')
else:
    print(f"Computing point-biserial correlations for {code_matrix_active.shape[1]:,} codes...")
    t0 = time.time()
    df_corr = compute_tier_correlations(code_matrix_active, y_all, code_tiers, active_code_indices)
    print(f"  Completed in {time.time()-t0:.1f}s")
    cache.save('correlation_results', df_corr)

print(f"\nCorrelation Analysis: {len(df_corr):,} codes")
print(f"{'Tier':<10} {'Codes':>8} {'Sig(p<.001)':>13} {'% Sig':>8} {'Mean|r|':>10} {'Max|r|':>10}")
print("-" * 65)
for tier in TIER_NAMES:
    sub = df_corr[df_corr['tier'] == tier]
    n = len(sub)
    n_sig = int(sub['significant_001'].sum())
    pct = n_sig / n * 100 if n > 0 else 0
    print(f"{tier:<10} {n:>8,} {n_sig:>13,} {pct:>7.1f}% "
          f"{sub['abs_correlation'].mean():>10.6f} {sub['abs_correlation'].max():>10.6f}")

total_sig = int(df_corr['significant_001'].sum())
print(f"\nOverall: {total_sig:,}/{len(df_corr):,} codes significantly correlated (p<0.001)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
tier_colors = {'common': '#2196F3', 'medium': '#4CAF50', 'rare': '#FF9800', 'tail': '#F44336'}

# 1. Proportion significant by tier and alpha level
ax = axes[0, 0]
tier_stats = []
for tier in TIER_NAMES:
    sub = df_corr[df_corr['tier'] == tier]
    for alpha, label in [(0.05, 'p<0.05'), (0.01, 'p<0.01'), (0.001, 'p<0.001')]:
        pct = (sub['p_value'] < alpha).mean() * 100
        tier_stats.append({'tier': tier, 'threshold': label, 'pct': pct})
df_sig = pd.DataFrame(tier_stats)
x = np.arange(len(TIER_NAMES))
for i, al in enumerate(['p<0.05', 'p<0.01', 'p<0.001']):
    vals = df_sig[df_sig['threshold'] == al]['pct'].values
    ax.bar(x + i * 0.25, vals, 0.22, label=al, alpha=0.85)
ax.set_xticks(x + 0.25)
ax.set_xticklabels(TIER_NAMES)
ax.set_ylabel('% of Codes Significant')
ax.set_title('Proportion of Codes Significantly Correlated with IP')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 2. Distribution of |correlation| by tier
ax = axes[0, 1]
for tier in TIER_NAMES:
    sub = df_corr[df_corr['tier'] == tier]
    ax.hist(sub['abs_correlation'], bins=50, alpha=0.5, label=tier, color=tier_colors[tier])
ax.set_xlabel('|Point-Biserial r|')
ax.set_ylabel('Count')
ax.set_title('Distribution of |Correlation| by Tier')
ax.legend()
ax.grid(alpha=0.3)

# 3. Mean |r| of top-20 per tier
ax = axes[1, 0]
top_r = []
for tier in TIER_NAMES:
    sub = df_corr[df_corr['tier'] == tier].nlargest(20, 'abs_correlation')
    top_r.append(sub['abs_correlation'].mean())
ax.bar(TIER_NAMES, top_r, color=[tier_colors[t] for t in TIER_NAMES])
ax.set_ylabel('Mean |r| of Top-20 Codes')
ax.set_title('Strongest Correlations by Tier (Top-20)')
ax.grid(axis='y', alpha=0.3)

# 4. Frequency ratio for significant codes
ax = axes[1, 1]
sig_codes = df_corr[df_corr['significant_001']]
for tier in TIER_NAMES:
    sub = sig_codes[sig_codes['tier'] == tier]
    if len(sub) > 0:
        ratios = np.clip(sub['freq_ratio'], 0, 10)
        ax.hist(ratios, bins=30, alpha=0.5,
                label=f"{tier} (n={len(sub)})", color=tier_colors[tier])
ax.set_xlabel('Frequency Ratio (pos/neg mean)')
ax.set_ylabel('Count')
ax.set_title('Code Frequency Ratio for Significant Codes (p<0.001)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/correlation_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

### B.2 Dimension Reduction

Three methods to reduce ~39k raw codes to `TARGET_DIM` features:
1. **PCA** (TruncatedSVD) — linear projection preserving variance
2. **AutoEncoder** — nonlinear compression via 4-GPU neural network
3. **SelectKBest(chi2)** — univariate filter on sparse data (replaces MI for speed)

In [ ]:
# B.2a: PCA (TruncatedSVD)
if cache.exists('pca_features'):
    print("Loading cached PCA features...")
    pca_features = cache.load('pca_features')
else:
    print(f"TruncatedSVD({TARGET_DIM}) on {code_matrix_active.shape}...")
    scaler = MaxAbsScaler()
    scaled = scaler.fit_transform(code_matrix_active)
    svd = TruncatedSVD(n_components=TARGET_DIM, random_state=RANDOM_STATE)
    pca_features = svd.fit_transform(scaled)
    print(f"  Explained variance: {svd.explained_variance_ratio_.sum():.4f}")
    cache.save('pca_features', pca_features)

print(f"PCA features: {pca_features.shape}")

In [ ]:
# B.2b: AutoEncoder (replaces UMAP — nonlinear but GPU-accelerated and tractable)
if cache.exists('ae_features'):
    print("Loading cached autoencoder features...")
    ae_features = cache.load('ae_features')
else:
    # log1p stabilizes scale for count data while preserving sparsity
    code_matrix_log = code_matrix_active.copy()
    code_matrix_log.data = np.log1p(code_matrix_log.data)

    print(f"Training autoencoder: {code_matrix_log.shape[1]} -> {TARGET_DIM}")
    ae_model = train_autoencoder(code_matrix_log, bottleneck_dim=TARGET_DIM)

    print("Encoding all members...")
    ae_features = encode_all(ae_model, code_matrix_log)

    raw_model = ae_model.module if hasattr(ae_model, 'module') else ae_model
    cache.save('ae_model', raw_model.state_dict())
    cache.save('ae_features', ae_features)

print(f"AutoEncoder features: {ae_features.shape}")

In [ ]:
# B.2c: SelectKBest with chi2 (sparse-native, replaces MI for speed)
# chi2 operates directly on sparse CSR — no dense conversion, O(n_features) not O(n*d*k).
if cache.exists('selectk_features') and cache.exists('selectk_indices'):
    print("Loading cached SelectKBest features...")
    selectk_features = cache.load('selectk_features')
    selectk_code_indices = cache.load('selectk_indices')
else:
    print(f"SelectKBest(chi2, k={TARGET_DIM}) on sparse {code_matrix_active.shape}...")
    t0 = time.time()

    selector = SelectKBest(score_func=chi2, k=TARGET_DIM)
    selectk_raw = selector.fit_transform(code_matrix_active, y_all)

    selectk_features = selectk_raw.toarray() if sparse.issparse(selectk_raw) else np.asarray(selectk_raw)
    selectk_code_indices = active_code_indices[selector.get_support()]

    print(f"  Completed in {time.time()-t0:.1f}s")
    cache.save('selectk_features', selectk_features)
    cache.save('selectk_indices', selectk_code_indices)
    cache.save('selectk_scores', selector.scores_)

print(f"SelectKBest features: {selectk_features.shape}")

# Show tier composition of selected codes
sel_tiers = [code_tiers.get(int(c), 'unknown') for c in selectk_code_indices]
sel_tier_counts = Counter(sel_tiers)
print(f"Selected codes by tier: {dict(sel_tier_counts)}")

### B.3 Downstream CatBoost Evaluation

Same pipeline as `moe_flashattn_3_lob3_downstream_running.py`:
CatBoost (2500 iter, depth=7, balanced), 10:1 downsample, train/val/test/OOT splits.

In [ ]:
all_results = {}

# --- 1. PCA ---
pca_names = [f'pca_{i}' for i in range(TARGET_DIM)]
all_results['pca_256'] = evaluate_feature_set(
    pca_features, pca_names, df_merged_base, f"PCA({TARGET_DIM})")

# --- 2. AutoEncoder ---
ae_names = [f'ae_{i}' for i in range(TARGET_DIM)]
all_results['ae_256'] = evaluate_feature_set(
    ae_features, ae_names, df_merged_base, f"AutoEncoder({TARGET_DIM})")

# --- 3. SelectKBest(chi2) ---
sel_names = [f'sel_{c}' for c in selectk_code_indices]
all_results['selectk_256'] = evaluate_feature_set(
    selectk_features, sel_names, df_merged_base, f"SelectKBest({TARGET_DIM}, chi2)")

In [ ]:
# --- 4. TE Embedding ---
_te_loaded = False
if cache.exists('te_embeddings'):
    print("Loading cached TE embeddings...")
    df_te = cache.load('te_embeddings')
    _te_loaded = True
else:
    try:
        print("TE embeddings not cached. Loading from BigQuery...")
        _client = bigquery.Client()
        df_te = _client.query(f"SELECT * FROM `{TE_EMBEDDING_TABLE}`").to_dataframe()
        cache.save('te_embeddings', df_te)
        _te_loaded = True
    except Exception as e:
        print(f"Could not load TE embeddings: {e}")

if _te_loaded:
    df_te['individual_id'] = df_te['individual_id'].astype(str)
    emb_cols = sorted(
        [c for c in df_te.columns if c.startswith('embedding_')],
        key=lambda c: int(c.split('_')[1]),
    )

    te_merge = df_merged_base[[
        'individual_id', 'index_dt', TARGET_COLUMN, 'ind_id_last_digit'
    ]].merge(df_te[['individual_id'] + emb_cols], on='individual_id', how='inner')
    te_merge = te_merge.drop_duplicates(
        subset=['individual_id', 'index_dt'], keep='last'
    )

    te_features = te_merge[emb_cols].values.astype(np.float32)
    te_base = te_merge[[
        'individual_id', 'index_dt', TARGET_COLUMN, 'ind_id_last_digit'
    ]].reset_index(drop=True)
    te_names = [f'te_{i}' for i in range(len(emb_cols))]

    print(f"TE matched: {len(te_merge):,} rows, {len(emb_cols)} dims")
    all_results['te_embedding'] = evaluate_feature_set(
        te_features, te_names, te_base, f"TE Embedding({len(emb_cols)})")
else:
    print("Skipping TE evaluation — embeddings unavailable.")

print(f"\nAll {len(all_results)} evaluations complete.")

### B.4 Comparison Results

In [ ]:
comp_rows = []
for name, result in all_results.items():
    row = {'representation': result['feature_set'], 'n_features': result['n_features']}
    for split in ['val', 'test', 'oot', 'oot_strict']:
        for metric in ['auc_roc', 'auc_pr', 'brier', 'lift_1pct', 'lift_5pct',
                       'lift_10pct', 'tp_1pct', 'precision_1pct',
                       'n_samples', 'n_positives', 'prevalence']:
            k = f'{split}_{metric}'
            if k in result:
                row[k] = result[k]
    comp_rows.append(row)

df_comparison = pd.DataFrame(comp_rows)

print("=" * 90)
print("DOWNSTREAM COMPARISON: Dimension Reduction Methods vs TE")
print("=" * 90)

display_cols = [
    'representation', 'n_features',
    'oot_strict_auc_roc', 'oot_strict_auc_pr', 'oot_strict_brier',
    'oot_strict_lift_1pct', 'oot_strict_lift_5pct', 'oot_strict_lift_10pct',
]
existing = [c for c in display_cols if c in df_comparison.columns]
print(df_comparison[existing].to_string(index=False))

cache.save('comparison_results', df_comparison)
print(f"\nSaved to {OUTPUT_DIR}/")

### B.5 Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

rep_names = df_comparison['representation'].values
short_names = [n.split('(')[0].strip() for n in rep_names]
colors = plt.cm.Set2(np.linspace(0, 1, len(rep_names)))

# AUC-ROC: test vs OOT-strict
ax = axes[0]
x = np.arange(len(rep_names))
width = 0.35
for i, split in enumerate(['test', 'oot_strict']):
    col = f'{split}_auc_roc'
    if col in df_comparison.columns:
        ax.bar(x + i * width, df_comparison[col].fillna(0), width,
               label=split, alpha=0.85)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC-ROC: Test vs OOT-Strict')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Lift@1%
ax = axes[1]
col = 'oot_strict_lift_1pct'
if col in df_comparison.columns:
    ax.barh(short_names, df_comparison[col].fillna(0), color=colors)
    ax.set_xlabel('Lift @ 1%')
    ax.set_title('Lift@1% (OOT-Strict)')
    ax.grid(axis='x', alpha=0.3)

# Dimensionality vs performance
ax = axes[2]
col = 'oot_strict_auc_roc'
if col in df_comparison.columns:
    for i, (_, row) in enumerate(df_comparison.iterrows()):
        ax.scatter(row['n_features'], row[col], s=150, c=[colors[i]],
                   label=short_names[i], zorder=5)
    ax.set_xlabel('Features (log scale)')
    ax.set_ylabel('AUC-ROC (OOT-Strict)')
    ax.set_title('Dimensionality vs Performance')
    ax.set_xscale('log')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/representation_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("=" * 90)
print("EXPERIMENT SUMMARY")
print("Raw Code Dimension Reduction vs Transformer Embedding")
print("=" * 90)

print(f"""
METHODOLOGY:
- Task: Commercial IP (inpatient) 6-month prediction
- Model: CatBoost (2500 iter, depth=7, balanced weights)
- Splits: Train (digits 0-7), Val (8), Test (9), OOT (>{OOT_CUTOFF_DATE}), OOT_strict (OOT+9)
- Downsample: {NEGATIVE_DOWNSAMPLE_RATIO}:1 neg:pos on train
- Primary metric: AUC-ROC on OOT_strict

REPRESENTATIONS COMPARED:
1. PCA({TARGET_DIM}): TruncatedSVD on {code_matrix_active.shape[1]:,} raw codes
2. AutoEncoder({TARGET_DIM}): {AE_HIDDEN_DIMS} hidden, 4-GPU trained, log1p input
3. SelectKBest({TARGET_DIM}): Top codes by chi-squared statistic (sparse-native)
4. TE Embedding({TARGET_DIM}): Pretrained transformer (R6 best, ASL + dense sampler)
""")

print("DOWNSTREAM RESULTS:")
existing = [c for c in display_cols if c in df_comparison.columns]
print(df_comparison[existing].to_string(index=False))

print(f"\nCORRELATION ANALYSIS:")
print(f"Thresholds: p20={freq_percentiles[0]:.0f}, p50={freq_percentiles[1]:.0f}, p80={freq_percentiles[2]:.0f}")
for tier in TIER_NAMES:
    sub = df_corr[df_corr['tier'] == tier]
    n_sig = int(sub['significant_001'].sum())
    pct = n_sig / len(sub) * 100 if len(sub) > 0 else 0
    print(f"  {tier}: {n_sig:,}/{len(sub):,} sig ({pct:.1f}%), "
          f"mean|r|={sub['abs_correlation'].mean():.6f}")

print("""
INTERPRETATION:
- PCA/AE/SelectK >= TE: TE adds no value beyond mechanical compression
- TE >> PCA: TE captures meaningful clinical semantics
- Common codes dominate correlation: aligns with TE common-code attention pattern
- Rare/tail codes weakly correlated: limited signal may explain TE under-representation
""")